# Distil BioCLIP-2 into a deployable ViT-B

Closes the 19pp coverage gap between the encoder that scores (BioCLIP-2, a 304M
ViT-L that cannot ship) and the one that fits a phone (BioCLIP v1, 46 MB at int4).

The teacher never runs here — its embeddings are already cached — so this is a
regression onto vectors, not a two-model training job.

**This notebook only trains.** Embedding and scoring happen on the machine that
already holds the data; the only thing that travels back is a checkpoint. That
keeps the upload to 3.9 GB instead of 9.6 GB, and it means the iNaturalist
photographs — which the student is *forbidden* to see — are never even present.

**Runtime → Change runtime type → A100** before running.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 1. Get the code and the bundle here

On your machine:

```bash
python -m plantid.train.pack_transfer --out distil_bundle.tar   # 3.9 GB
```

Put `distil_bundle.tar` and the repo on Drive, then:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO   = '/content/drive/MyDrive/plantid'          # adjust
BUNDLE = '/content/drive/MyDrive/distil_bundle.tar'  # adjust

# copy the repo to local disk — training off Drive is slow
!mkdir -p /content/work && cp -r $REPO/plantid $REPO/notebooks /content/work/ 2>/dev/null
%cd /content/work
!tar xf $BUNDLE          # unpacks into data/processed/
!du -sh data/processed


In [ ]:
!pip -q install open_clip_torch


## 2. Check the transfer set before training anything

The correctness check that matters. The student must never see an image the
system is evaluated on: not the catalogue test split, not the val split (where
temperature scaling is fitted), and no iNaturalist observation.

Expect **48,564 images** and **`any iNat: False`**. If the count differs, stop —
the bundle is wrong and any result would be uninterpretable.


In [ ]:
import sys; sys.path.insert(0, '.')
from plantid.train.distil import build_transfer_set

df = build_transfer_set()
print(f'transfer set: {len(df):,} images, teacher dim {df.teacher.iloc[0].shape[0]}')
print('catalogue :', sum(p.startswith('images/') for p in df.local_path))
print('background:', sum(p.startswith('images_background/') for p in df.local_path))
print('any iNat  :', any('images_inat' in p for p in df.local_path))
assert len(df) == 48564 and not any('images_inat' in p for p in df.local_path)


## 3. Train

~48k images at ~700 img/s is roughly a minute per epoch, so 40 epochs is under an
hour. Checkpoints are written every epoch, so a disconnect costs at most one.


In [ ]:
!python -m plantid.train.distil \
    --epochs 40 --batch-size 256 --lr 1e-4 --head-lr 1e-3 \
    --workers 8 --out data/processed/distil/student.pt


## 4. Read the curves before believing the number

**If train cosine keeps climbing while held-out flattens, the student is
memorising 48k images.** The fix is a larger transfer set — PlantNet has ~244k
images not yet downloaded — not more epochs. A held-out curve still rising at
epoch 40 means it is worth training longer.


In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open('data/processed/distil/student.json'))
ep = [r['epoch'] for r in h['history']]
plt.figure(figsize=(7,4))
plt.plot(ep, [r['train_cos'] for r in h['history']], label='train')
plt.plot(ep, [r['val_cos'] for r in h['history']], label='held out')
plt.axhline(h['baseline_cos'], ls='--', c='grey', label='at init')
plt.xlabel('epoch'); plt.ylabel('cosine to teacher')
plt.legend(); plt.grid(alpha=.3); plt.show()
print('best held-out cosine:', max(r['val_cos'] for r in h['history']))


## 5. Bring the checkpoint home


In [ ]:
!cp data/processed/distil/student.pt $REPO/  # ~350 MB
!ls -la $REPO/student.pt


## 6. Score it locally

Back on the machine with the full dataset. `bioclip1_distil` is registered in
`ENCODERS`, so nothing else changes:

```bash
mkdir -p data/processed/distil && mv student.pt data/processed/distil/

PYTHONPATH=. python -c "
from plantid.features import embed_catalog, embed_background, embed_inat
for m in (embed_catalog, embed_background, embed_inat):
    m.main(variant='bioclip1_distil')"          # ~11 min on an M4 Max

PYTHONPATH=. python -m plantid.eval.rejection --variant bioclip1_distil
```

### The bar

| | genus | 95% CI | species | precision @20% | coverage |
|---|---|---|---|---|---|
| BioCLIP-2 (teacher, cannot ship) | 0.9747 | [0.9653, 0.9827] | 0.8460 | 0.956 | 0.722 |
| BioCLIP v1 (ships today) | 0.9310 | [0.9163, 0.9440] | 0.7604 | 0.946 | 0.531 |
| **distilled student** | ? | | | | |

**Pass = beats BioCLIP v1 by a margin whose paired interval excludes zero**, using
the same cluster bootstrap as everything else in this project:

```python
from plantid.config import DATA_PROCESSED as D
from plantid.eval.rejection import build_observations, cluster_bootstrap
F = {v: build_observations(str(D/f'inat_{v}.npz'), variant=v)[0]
     for v in ('bioclip1', 'bioclip1_distil')}
m  = F['bioclip1'].in_catalog.values
sp = F['bioclip1'].species.values[m]
for col in ('genus_ok', 'species_ok'):
    d = (F['bioclip1_distil'][col].values[m].astype(float)
         - F['bioclip1'][col].values[m].astype(float))
    lo, hi = cluster_bootstrap(d, sp)
    print(f'{col}: {d.mean():+.4f}  [{lo:+.4f}, {hi:+.4f}]')
```

If it passes, the rest of the path is already measured: Core ML int4 export costs
~1.3pp of genus accuracy, and `computeUnits` must be pinned to
`.cpuAndNeuralEngine` or the GPU backend returns garbage (`ONDEVICE_FINDINGS.md`).

If it fails, the honest conclusion is that feature distillation on 48k images
does not close this gap, and the remaining options are raising the size budget
for BioCLIP-2 (~164 MB at int4) or accepting ~50% coverage.
